In [1]:
import os
from datasets import load_dataset

print(os.getcwd())
print(os.listdir())

dataset = load_dataset("csv", data_files="./support_tickets_400.csv")

C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\ticker triager\ticket_triager\ml
['.ipynb_checkpoints', 'inference.py', 'results', 'support_tickets_400.csv', 'synthetic_data.py', 'tickets_test.csv', 'tickets_train.csv', 'ticket_lora_adapter', 'Untitled.ipynb', '__pycache__']


In [2]:
import huggingface_hub

print(huggingface_hub.__version__)

1.30.0


In [3]:
from huggingface_hub import model_info

info = model_info("distilbert/distilbert-base-uncased")
print(info.id)

distilbert/distilbert-base-uncased


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8)  # 8 categories

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 6440.59it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# from peft import LoraConfig, get_peft_model, TaskType

# lora_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,
#     r=8,               # rank — smaller = fewer trainable params
#     lora_alpha=16,
#     lora_dropout=0.1,
#     target_modules=["q_lin", "v_lin"],  # DistilBERT's attention layers
# )

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()
# Output will show something like: trainable params: 0.5M || all params: 66M || trainable%: 0.7%

In [6]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["classifier", "pre_classifier"],  # <-- add this line
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 744,200 || all params: 67,703,824 || trainable%: 1.0992


In [7]:
# Define your label mapping once
label_list = ["billing", "account", "orders", "shipping", "refund", "technical", "product", "subscription"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

def tokenize(batch):
    texts = [s + " " + b for s, b in zip(batch["subject"], batch["body"])]
    encoded = tokenizer(texts, truncation=True, padding="max_length")
    encoded["labels"] = [label2id[cat] for cat in batch["category"]]
    return encoded

tokenized = dataset.map(tokenize, batched=True)

from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"])
trainer.train()

Map: 100%|██████████████████████████████████████████████████████████████████| 400/400 [00:00<00:00, 1106.97 examples/s]
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.081222
20,2.040281
30,1.975410
40,1.865799
50,1.715315
60,1.432345
70,1.115084
80,0.877632
90,0.725638
100,0.634636


C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=150, training_loss=1.096121416091919, metrics={'train_runtime': 2019.7066, 'train_samples_per_second': 0.594, 'train_steps_per_second': 0.074, 'total_flos': 161721306316800.0, 'train_loss': 1.096121416091919, 'epoch': 3.0})

In [8]:
model.save_pretrained("ticket_lora_adapter")
tokenizer.save_pretrained("ticket_lora_adapter")

('ticket_lora_adapter\\tokenizer_config.json',
 'ticket_lora_adapter\\tokenizer.json')

In [9]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=8)
model = PeftModel.from_pretrained(base_model, "ticket_lora_adapter")
tokenizer = AutoTokenizer.from_pretrained("ticket_lora_adapter")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 2438.96it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="./support_tickets_400.csv")

# dataset["train"] currently holds everything — split it
split_dataset = dataset["train"].train_test_split(test_size=0.15, seed=42)

# Now you have both splits
dataset = split_dataset  # dataset["train"] and dataset["test"]

In [11]:
tokenized = dataset.map(tokenize, batched=True)

Map: 100%|█████████████████████████████████████████████████████████████████████| 60/60 [00:00<00:00, 982.05 examples/s]


In [12]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("./support_tickets_400.csv")
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df["category"], random_state=42)
train_df.to_csv("tickets_train.csv", index=False)
test_df.to_csv("tickets_test.csv", index=False)

In [13]:
dataset = load_dataset("csv", data_files={"train": "tickets_train.csv", "test": "tickets_test.csv"})

Generating train split: 340 examples [00:00, 22648.87 examples/s]
Generating test split: 60 examples [00:00, 5001.26 examples/s]


In [14]:
predictions = trainer.predict(tokenized["test"])

C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [15]:
predictions

PredictionOutput(predictions=array([[-1.56918859e+00,  3.17635012e+00, -9.80404019e-01,
        -7.47766644e-02, -2.77740812e+00,  1.43829691e+00,
        -1.34593916e+00, -8.76594424e-01],
       [-5.77707410e-01, -7.54903257e-02, -2.42598295e+00,
        -1.85492206e+00, -1.38190556e+00,  4.24097478e-01,
        -1.67542088e+00,  3.70034838e+00],
       [ 1.17976737e+00, -1.59689081e+00, -1.93813276e+00,
        -1.73972607e+00,  4.12867212e+00, -2.16731095e+00,
        -1.67069817e+00, -8.61306489e-01],
       [ 3.13693476e+00, -5.40035129e-01, -1.38806355e+00,
        -1.15755248e+00,  1.55808091e+00, -2.56061840e+00,
        -2.32934952e+00, -1.04555285e+00],
       [-1.95050383e+00,  9.64652181e-01, -7.47139931e-01,
         1.81628633e+00, -2.42616129e+00,  1.25767362e+00,
        -5.29565692e-01, -1.56010926e+00],
       [-2.18283653e+00, -2.89393246e-01, -9.12885785e-01,
        -2.28075117e-01, -2.12292194e+00,  2.51902628e+00,
         8.99089217e-01, -2.73355693e-01],
     

In [16]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

In [17]:
from sklearn.metrics import classification_report, accuracy_score

acc = accuracy_score(labels, preds)
print(f"Accuracy: {acc:.2%}")

print(classification_report(labels, preds, target_names=label_list))

Accuracy: 98.33%
              precision    recall  f1-score   support

     billing       1.00      1.00      1.00         3
     account       1.00      1.00      1.00         7
      orders       1.00      1.00      1.00         3
    shipping       1.00      1.00      1.00         6
      refund       0.90      1.00      0.95         9
   technical       1.00      1.00      1.00        14
     product       1.00      1.00      1.00         9
subscription       1.00      0.89      0.94         9

    accuracy                           0.98        60
   macro avg       0.99      0.99      0.99        60
weighted avg       0.98      0.98      0.98        60



In [18]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# ---- paths ----
MODEL_PATH = "ticket_lora_adapter"   # folder where you saved the trained model
BASE_MODEL = "distilbert-base-uncased"

label_list = [
    "billing", "account", "orders", "shipping",
    "refund", "technical", "product", "subscription",
]
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# ---- load tokenizer + model from saved folder ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

# ---- prediction function ----
def predict(subject: str, body: str):
    text = f"{subject} {body}"
    inputs = tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        confidence, pred_id = torch.max(probs, dim=1)

    category = model.config.id2label[pred_id.item()]
    return category, confidence.item()

# ---- run on sample tickets ----
sample_tickets = [
    ("Payment failed", "My card was charged twice for the same order and I need a refund."),
    ("Can't login", "I keep entering my password correctly but it says invalid credentials."),
    ("Package never arrived", "Tracking says delivered but I never received my order."),
    ("App keeps crashing", "The mobile app crashes every single time I try to open the dashboard."),
    ("Cancel my subscription", "I want to cancel my subscription before the next billing cycle."),
]

print(f"{'Subject':30} | {'Predicted Category':20} | Confidence")
print("-" * 70)
for subject, body in sample_tickets:
    category, confidence = predict(subject, body)
    print(f"{subject:30} | {category:20} | {confidence:.2%}")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 3477.37it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Subject                        | Predicted Category   | Confidence
----------------------------------------------------------------------
Payment failed                 | refund               | 67.67%
Can't login                    | account              | 75.74%
Package never arrived          | shipping             | 76.81%
App keeps crashing             | technical            | 77.19%
Cancel my subscription         | subscription         | 83.87%


In [20]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [21]:
label_list_urgency = ["low", "medium", "high"]
label2id_urgency = {label: i for i, label in enumerate(label_list_urgency)}
id2label_urgency = {i: label for label, i in label2id_urgency.items()}

def tokenize_urgency(batch):
    texts = [s + " " + b for s, b in zip(batch["subject"], batch["body"])]
    encoded = tokenizer(texts, truncation=True, padding="max_length", max_length=128)
    encoded["labels"] = [label2id_urgency[u] for u in batch["urgency"]]
    return encoded

tokenized_urgency = dataset.map(tokenize_urgency, batched=True)

model_urgency = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=id2label_urgency, label2id=label2id_urgency
)

lora_config_urgency = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["classifier", "pre_classifier"],
)
model_urgency = get_peft_model(model_urgency, lora_config_urgency)

trainer_urgency = Trainer(
    model=model_urgency, args=args,
    train_dataset=tokenized_urgency["train"],
    eval_dataset=tokenized_urgency["test"],
    compute_metrics=compute_metrics,
)
trainer_urgency.train()

model_urgency.save_pretrained("urgency_lora_adapter")
tokenizer.save_pretrained("urgency_lora_adapter")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 6210.93it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775

Step,Training Loss
10,1.112554
20,1.094060
30,1.091810
40,1.074989
50,1.071523
60,1.084649
70,1.102417
80,1.067871
90,1.058617
100,1.037105


C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in distilbert-base-uncased.
  warnings.warn(
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in distilbert-base-uncased - will assume that the vocabulary was not mo

('urgency_lora_adapter\\tokenizer_config.json',
 'urgency_lora_adapter\\tokenizer.json')

In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

BASE_MODEL = "distilbert-base-uncased"

CATEGORY_MODEL_PATH = "ticket_lora_adapter"
URGENCY_MODEL_PATH = "urgency_lora_adapter"

category_list = [
    "billing", "account", "orders", "shipping",
    "refund", "technical", "product", "subscription",
]
urgency_list = ["low", "medium", "high"]

category_id2label = {i: l for i, l in enumerate(category_list)}
category_label2id = {l: i for i, l in enumerate(category_list)}
urgency_id2label = {i: l for i, l in enumerate(urgency_list)}
urgency_label2id = {l: i for i, l in enumerate(urgency_list)}

# ---- load category model ----
category_tokenizer = AutoTokenizer.from_pretrained(CATEGORY_MODEL_PATH)
category_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(category_list),
    id2label=category_id2label, label2id=category_label2id,
)
category_model = PeftModel.from_pretrained(category_base, CATEGORY_MODEL_PATH)
category_model.eval()

# ---- load urgency model ----
urgency_tokenizer = AutoTokenizer.from_pretrained(URGENCY_MODEL_PATH)
urgency_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(urgency_list),
    id2label=urgency_id2label, label2id=urgency_label2id,
)
urgency_model = PeftModel.from_pretrained(urgency_base, URGENCY_MODEL_PATH)
urgency_model.eval()


def predict(subject: str, body: str):
    text = f"{subject} {body}"

    cat_inputs = category_tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    urg_inputs = urgency_tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

    with torch.no_grad():
        cat_logits = category_model(**cat_inputs).logits
        cat_probs = torch.softmax(cat_logits, dim=1)
        cat_conf, cat_id = torch.max(cat_probs, dim=1)

        urg_logits = urgency_model(**urg_inputs).logits
        urg_probs = torch.softmax(urg_logits, dim=1)
        urg_conf, urg_id = torch.max(urg_probs, dim=1)

    category = category_model.config.id2label[cat_id.item()]
    urgency = urgency_model.config.id2label[urg_id.item()]

    return category, cat_conf.item(), urgency, urg_conf.item()


sample_tickets = [
    ("Payment failed", "My card was charged twice for the same order and I need a refund."),
    ("Can't login", "I keep entering my password correctly but it says invalid credentials."),
    ("Package never arrived", "Tracking says delivered but I never received my order."),
    ("App keeps crashing", "The mobile app crashes every single time I try to open the dashboard."),
    ("Cancel my subscription", "I want to cancel my subscription before the next billing cycle."),
]

print(f"{'Subject':28} | {'Category':14} | {'Conf':6} | {'Urgency':8} | {'Conf':6}")
print("-" * 80)
for subject, body in sample_tickets:
    category, cat_conf, urgency, urg_conf = predict(subject, body)
    print(f"{subject:28} | {category:14} | {cat_conf:.2%} | {urgency:8} | {urg_conf:.2%}")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 4737.19it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00

Subject                      | Category       | Conf   | Urgency  | Conf  
--------------------------------------------------------------------------------
Payment failed               | refund         | 67.67% | high     | 41.67%
Can't login                  | account        | 75.74% | high     | 40.58%
Package never arrived        | shipping       | 76.81% | high     | 38.46%
App keeps crashing           | technical      | 77.19% | high     | 36.75%
Cancel my subscription       | subscription   | 83.87% | high     | 41.89%
